In [11]:
import os
import sys
import platform
import subprocess
import shutil

In [3]:
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in str(get_ipython())

if IS_KAGGLE:
    print("Running on Kaggle - Internet must be ON")
elif IS_COLAB:
    print("Running on Colab")
else:
    print("Running locally on Windows\Linux")

Running locally on Windows\Linux


Colab ONLY: This will crash the kernel the first time. This is expected - if you rerun from the start, it will work.

In [4]:
if IS_COLAB:
    print("Running on Colab - Setting up Cloud environment...")
    # Colab only
    !apt-get install -y git-core ffmpeg espeak-ng
    !pip install -q condacolab
    import condacolab
    condacolab.install()
    condacolab.check()
else:
    print("Running locally \ Kaggle - using existing Conda environment.")

Running locally \ Kaggle - using existing Conda environment.


Clone repo

In [5]:
# Check if we are already inside the folder or if it's nearby
# Check if folder exists or if we are on Cloud (Kaggle/Colab)
if os.path.basename(os.getcwd()) == 'VoiceCraft':
    print("Already inside VoiceCraft folder.")
elif os.path.exists('VoiceCraft'):
    print("VoiceCraft folder found, entering...")
    # Always enter the project directory
    %cd VoiceCraft
else:
    print("Cloning repository for the first time...")
    !git clone https://github.com/jasonppy/VoiceCraft.git
    # Always enter the project directory
    %cd VoiceCraft

# Adding paths so Python can find the 'models' and 'src' folders
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
    
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Current working directory: {os.getcwd()}")

Already inside VoiceCraft folder.
Current working directory: /mnt/c/Users/Sukiennik/Desktop/Daniella/MSC/2026/A/Speech/Project/git/VoiceCraft


for local running only : install relvance tourch for GPU 

In [7]:
IS_CLOUD = os.path.exists('/kaggle/working') or 'google.colab' in str(get_ipython()) # Local or cloud?

# Checking if Torch is already installed and functional

torch_installed = False
try:
    import torch
    if torch.cuda.is_available():
        torch_installed = True
        print(f"✅ Torch {torch.__version__} with CUDA is ready.")
except:
    print("❌ Torch is not installed or not working correctly.")

if not torch_installed:
    if not IS_CLOUD:
        print("Installing Torch for Windows\ Linux (CUDA 11.8)...")
        !pip install torch==2.1.2+cu118 torchaudio==2.1.2+cu118 --index-url https://download.pytorch.org/whl/cu118
        import torch # reload after installtion 
    else:
        print("Cloud environment: Installing default Torch...")
        !pip install torch torchaudio
        import torch
else:
    print("Skipping Torch installation as it is already functional.")

✅ Torch 2.1.2+cu118 with CUDA is ready.
Skipping Torch installation as it is already functional.


In [8]:
# Main installation block
!conda install -y -c conda-forge montreal-forced-aligner=2.2.17 openfst=1.8.2 kaldi=5.5.1068
!pip install joblib==1.3.2 numpy==1.26.4 tensorboard datasets==2.16.0 torchmetrics==0.11.1 phonemizer==3.2.1
!pip install transformers==4.38.2 huggingface_hub==0.22.2

2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /home/sukiennik/miniconda3/envs/voicecraft_linux

  added / updated specs:
    - kaldi=5.5.1068
    - montreal-forced-aligner=2.2.17
    - openfst=1.8.2


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    adwaita-icon-theme-49.0    |           unix_0         617 KB  conda-forge
    aom-3.9.1                  |       hac33072_0         2.6 MB  conda-forge
    at-spi2-atk-2.38.0         |       h0630a04_3         332 KB  conda-forge
    at-spi2-core-2.40.3        |       h0630a04_0         643 KB  conda-forge
    atk-1.0-2.38.0             |      

In [9]:
!mfa version

/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/praatio/utilities/utils.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
2.2.17


In [ ]:
!pip install -e git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0#egg=audiocraft


In [3]:
import torch
import audiocraft
from audiocraft.models import MusicGen

print("--- Final System Check ---")
print(f"PyTorch using GPU: {torch.cuda.is_available()}")
print(f"AudioCraft Path: {audiocraft.__file__}")

try:
    # Small test to see if models can initialize
    model = MusicGen.get_pretrained('facebook/musicgen-small')
    print("✅ Success! MusicGen model loaded correctly.")
except Exception as e:
    print(f"❌ Still a small issue: {e}")

--- Final System Check ---
PyTorch using GPU: True
AudioCraft Path: /home/sukiennik/projects/VoiceCraft/src/audiocraft/audiocraft/__init__.py


/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


✅ Success! MusicGen model loaded correctly.


In [4]:
# Checking if the specific Meta AudioCraft version is already there
try:
    import audiocraft
except ImportError:
    import sys
    import os
    audiocraft_path = os.path.join(os.getcwd(), "src", "audiocraft")
    if os.path.exists(audiocraft_path):
        sys.path.append(audiocraft_path)
    
    # Try again
    try:
        import audiocraft
    except ImportError:
        # Installing Meta's Audiocraft
        print("❌ AudioCraft not found. Installing Meta's specific version...")
        !pip install -e git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0#egg=audiocraft

import audiocraft
print(f"✅ AudioCraft is ready! (Location: {audiocraft.__file__})")


✅ AudioCraft is ready! (Location: /home/sukiennik/projects/VoiceCraft/src/audiocraft/audiocraft/__init__.py)


In [5]:
try:
    # Checking if VoiceCraft models can be imported
    import models.voicecraft as voicecraft
    print("✅ VoiceCraft models found and ready!")
except ImportError as e:
    print(f"❌ Could not find VoiceCraft models: {e}")
    print("Tip: Check if you are in the correct directory.")

try:
    import audiocraft
    version = getattr(audiocraft, '__version__', 'Installed')
    print(f"✅ AudioCraft is ready! ({version})")
    # print(f"✅ AudioCraft is ready! (Version: {audiocraft.__version__})")
except ImportError:
    print("⚠️ AudioCraft not found in environment. Attempting to link from src...")
    if os.path.exists("src/audiocraft"):
        sys.path.append(os.path.abspath("src/audiocraft"))
        import audiocraft
        print(f"✅ AudioCraft linked from src! (Version: {audiocraft.__version__})")
    else:
        !pip install -e src/audiocraft
        import audiocraft
        print(audiocraft.__version__)

✅ VoiceCraft models found and ready!
✅ AudioCraft is ready! (1.0.0)


In [6]:
# Install xformers compatible with PyTorch 2.1.2 and CUDA 11.8
!pip install xformers==0.0.23.post1 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
# Installing the latest compatible version for CUDA 11.8 from the provided list
# !pip install --force-reinstall xformers==0.0.27.post2+cu118 --index-url https://download.pytorch.org/whl/cu118

# Installing synchronized versions for Windows + CUDA 11.8
# !pip install torch==2.1.2+cu118 torchaudio==2.1.2+cu118 xformers==0.0.23.post1 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
     ------ --------------------------------- 1.8/10.8 MB 9.1 MB/s eta 0:00:01
     --------------- ------------------------ 4.2/10.8 MB 10.5 MB/s eta 0:00:01
     ------------------------ --------------- 6.6/10.8 MB 10.6 MB/s eta 0:00:01
     --------------------------------- ------ 8.9/10.8 MB 10.9 MB/s eta 0:00:01
     ---------------------------------------- 10.8/10.8 MB 10.8 MB/s  0:00:00
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
     ---------------------------------------- 0.0/2.7 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.7 GB 11.2 MB/s eta 0:04:02
     ---------------------------------------- 0.0/2.7 GB 11.4 MB/s eta 0:03:56
     ---------------------------------------- 0.0/2.7 GB 11.5 MB/s eta 0:03:55
     ---------------------------------------- 0.0/2.7 GB 11.5 MB/s eta 0:03:54
     ----------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 2.16.0 requires fsspec[http]<=2023.10.0,>=2023.1.0, but you have fsspec 2025.12.0 which is incompatible.
gradio 3.50.2 requires numpy~=1.0, but you have numpy 2.2.6 which is incompatible.
torchaudio 2.7.1+cu118 requires torch==2.7.1+cu118, but you have torch 2.4.0+cu118 which is incompatible.


In [ ]:
# Installing the Intel runtime libraries that provide missing DLLs like fbgemm
# !pip install mkl mkl-include

   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/155.5 MB 1.1 MB/s eta 0:02:19
   ---------------------------------------- 0.8/155.5 MB 1.6 MB/s eta 0:01:38
   ---------------------------------------- 1.8/155.5 MB 2.5 MB/s eta 0:01:02
    --------------------------------------- 2.1/155.5 MB 2.0 MB/s eta 0:01:16
    --------------------------------------- 2.9/155.5 MB 2.3 MB/s eta 0:01:07
    --------------------------------------- 3.4/155.5 MB 2.4 MB/s eta 0:01:03
   - -------------------------------------- 4.2/155.5 MB 2.6 MB/s eta 0:00:59
   - -------------------------------------- 5.0/155.5 MB 2.8 MB/s eta 0:00:55
   - ----------------

In [7]:
print(f"Torch version: {torch.__version__}")
print(f"Is CUDA available? {torch.cuda.is_available()}")
print("If you see True, the GPU is ready!")

Torch version: 2.1.2+cu118
Is CUDA available? True
If you see True, the GPU is ready!


In [8]:
import xformers
import audiocraft
print("Success! Everything is connected.")

Success! Everything is connected.


In [9]:
!mfa model download dictionary english_us_arpa && \
mfa model download acoustic english_us_arpa

/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/praatio/utilities/utils.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
 WARNING  Local version of model already exists                                 
          (/home/sukiennik/Documents/MFA/pretrained_models/dictionary/english_us
          _arpa.dict). Use the --ignore_cache flag to force redownloading.      
/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/praatio/utilities/utils.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_

In [10]:
print("🚀 Environment is set and ready.")

🚀 Environment is set and ready.


Environment setup

In [ ]:
# # Environment setup

# conda create -n voicecraft python=3.9.16
# conda activate voicecraft

# pip install -e git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0#egg=audiocraft
# pip install xformers==0.0.22
# pip install torchaudio==2.0.2 torch==2.0.1 # this assumes your system is compatible with CUDA 11.7, otherwise checkout https://pytorch.org/get-started/previous-versions/#v201
# apt-get install ffmpeg # if you don't already have ffmpeg installed
# apt-get install espeak-ng # backend for the phonemizer installed below
# pip install tensorboard==2.16.2
# pip install phonemizer==3.2.1
# pip install datasets==2.16.0
# pip install torchmetrics==0.11.1
# pip install huggingface_hub==0.22.2
# # install MFA for getting forced-alignment, this could take a few minutes
# conda install -c conda-forge montreal-forced-aligner=2.2.17 openfst=1.8.2 kaldi=5.5.1068
# # install MFA english dictionary and model
# mfa model download dictionary english_us_arpa
# mfa model download acoustic english_us_arpa
# # pip install huggingface_hub
# # conda install pocl # above gives an warning for installing pocl, not sure if really need this

# # to run ipynb
# conda install -n voicecraft ipykernel --no-deps --force-reinstall


In [ ]:
# Installing VoiceCraft in editable mode so changes take effect immediately
# !pip install -e .